In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_18022/3003301750.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data_atlas = pd.read_csv(


In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 101  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

lst_sigma_tot_born = []
lst_sqrt_s = []
lst_error = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0732

max_sqrt_s = 13000
step = 100
n_points = 10000

model_params = {
    'atlas': {
        'pl':  {'mg': 0.417, 'a1': 1.563, 'a2': 2.22}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}

lst_amp_born = []
lst_sqrt_s = []
lst_sigma_tot_born = []




In [4]:
# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


# -------------------------------
# Inner integral (over phi)
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) - 
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n_points)
    return result

# -------------------------------
# Outer integral (over k)
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    return phi_integral(k, mg, a1, a2, m2_func, q, n_points)

# -------------------------------
# Double integral computation
# -------------------------------
def compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points=10000):
    result, _ = fixed_quad(
        lambda k: k_integral(k, mg, a1, a2, m2_func, q, n_points),
        0, sqrt_s_val, 
        n=n_points
    )
    return result

def born_amp(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot_born(amp_born_value, s):
    return amp_born_value.imag / s * 0.389379323


In [5]:

def calculate_born_cross_sections(start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points=10000):
    """Calculate cross sections for all sqrt_s values"""
    
    # Generate array of sqrt_s values
    sqrt_s_values = np.arange(start_sqrt_s, max_sqrt_s + step, step)
    
    # Process each sqrt_s value
    lst_sigma_tot_born = []
    lst_sqrt_s = []
    lst_amp_born = []
    
    for sqrt_s_val in sqrt_s_values:
        # Compute the double integral
        q = 0  # Assuming q=0 as in the original code
        diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points)
        
        # Calculate amplitude and cross section
        s = sqrt_s_val * sqrt_s_val
        amp_born_value = born_amp(diff_T, s, epsilon, 0)
        sigma_tot_born_value = sigma_tot_born(amp_born_value, s)
        
        # Store results
        lst_sigma_tot_born.append(sigma_tot_born_value)
        lst_sqrt_s.append(sqrt_s_val)
        lst_amp_born.append(amp_born_value)
    
    return lst_sigma_tot_born, lst_sqrt_s, lst_amp_born



In [6]:
mass_model = 'pl'
ensemble = 'atlas'
m2_func = get_m2_function(mass_model)
params = model_params[ensemble][mass_model]
mg, a1, a2 = params['mg'], params['a1'], params['a2']
epsilon = epsilon_values[ensemble]

# Calculate all cross sections
lst_sigma_tot_born, lst_sqrt_s, lst_amp_born = calculate_born_cross_sections(
    start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points
)

lst_s = [val**2 for val in lst_sqrt_s]

In [7]:
n = 10
q_max = 0.1
chi_list = []

# -------------------------------
# Full chi(s,b) including k, phi, and new q integral
# -------------------------------
def chi_integral(sqrt_s_values, mg, a1, a2, m2_func, epsilon, b):
    """
    Computes chi(s,b) = (1/s) ∫_0^qmax q dq J0(b q) [i 8 s^(1+ε) (∫_0^√s k dk ∫_0^2π dφ (T1-T2)) ]
    """
    # chi_list = []

    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        # integrand over q
        def q_integral(q):
            t = -q**2
            # compute existing double integral over k and phi
            diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q)
            return (q * j0(b * q) * born_amp(diff_T, s, epsilon, t))/s

        # integrate real and imaginary parts separately
        real_part, _ = fixed_quad(lambda q: np.vectorize(q_integral)(q).real, 0, q_max, n=n)
        imag_part, _ = fixed_quad(lambda q: np.vectorize(q_integral)(q).imag, 0, q_max, n=n)

        chi_list.append((real_part + 1j * imag_part))

    return chi_list


In [8]:
amp_list = []
b_max = 10

# -------------------------------
# Eikonal amplitude A_eik(s,t)
# -------------------------------
def eikonal_amplitude(sqrt_s_values, mg, a1, a2, m2_func, epsilon):


    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        # integrand over b
        def b_integrand(b):
            chi_val = chi_integral([sqrt_s_val], mg, a1, a2, m2_func, epsilon, b)[0]
            return b * (1 - np.exp(1j * chi_val))

        # integrate real and imaginary parts separately
        real_part, _ = fixed_quad(lambda b: np.vectorize(b_integrand)(b).real, 0, b_max, n=n)
        imag_part, _ = fixed_quad(lambda b: np.vectorize(b_integrand)(b).imag, 0, b_max, n=n)

        A_eik = 1j * s * (real_part + 1j * imag_part)
        amp_list.append(A_eik)

    return amp_list


# Example usage
A_eik_values = eikonal_amplitude(lst_sqrt_s, mg, a1, a2, m2_func, epsilon)
print(A_eik_values)


[237892.53891588992j, 942171.9894854297j, 2112861.672220228j, 3749961.587120284j, 5853471.734185598j, 8423392.11341617j, 11459722.724812001j, 14962463.56837309j, 18931614.644099437j, 23367175.951991044j, 28269147.492047906j, 33637529.26427003j, 39472321.26865741j, 45773523.50521004j, 52541135.97392794j, 59775158.674811095j, 67475591.6078595j, 75642434.77307318j, 84275688.1704521j, 93375351.79999629j, 102941425.66170573j, 112973909.75558044j, 123472804.0816204j, 134438108.6398256j, 145869823.4301961j, 157767948.45273185j, 170132483.70743284j, 182963429.19429907j, 196260784.91333058j, 210024550.86452737j, 224254727.04788938j, 238951313.46341667j, 254114310.11110923j, 269743716.99096704j, 285839534.1029901j, 302401761.44717836j, 319430399.023532j, 336925446.8320508j, 354886904.8727349j, 373314773.1455842j, 392209051.6505988j, 411569740.3877787j, 431396839.35712385j, 451690348.5586342j, 472450267.99230987j, 493676597.6581508j, 515369337.55615693j, 537528487.6863283j, 560154048.048665j, 583

In [9]:
def sigma_tot_eik(amp, s):
    return (4*np.pi)/s * amp.imag * 0.389379323   

lst_sigma_tot_eik = [sigma_tot_eik(amp, s) for amp, s in zip(A_eik_values, lst_s)]
print(lst_sigma_tot_eik)


[114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.1092427999047, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.1092427999047, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279990473, 114.10924279

In [10]:
def add_iterative_curve(fig, x_data, y_data, 
                        curve_name:str=None, color:str='blue', line_type:str='lines+markers'):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode=line_type, 
    name=curve_name,
    line=dict(
        color=color,
        width=2),
    marker=dict(size=4))
)
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')


fig_sigma = go.Figure()

add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_born, curve_name='sigma tot born')
add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_eik, curve_name='sigma tot eikonal', color='red')


# Add ATLAS data
fig_sigma.add_trace(go.Scatter(
    x=x_atlas,
    y=y_atlas,
    mode='markers',
    marker=dict(
        color='black',
        size=6,
        symbol='square'
    ),
    error_y=dict(
        type='data',
        array=y_error_atlas,
        visible=True
    ),
    name='ATLAS Data'
))

# Configure layout
fig_sigma.update_layout(
    title='Sigma Tot vs. sqrt(s) - b = [0, 10], q = [0, 0.1]',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Sigma Tot [mb]',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_sigma.update_xaxes(gridcolor='lightgray')
fig_sigma.update_yaxes(gridcolor='lightgray')

fig_sigma.show(renderer = 'browser')

Opening in existing browser session.
